# skforecast-ai — Demo Phase 8

This notebook demonstrates **Phase 8: ForecastingAssistant (Public API)**.

The `ForecastingAssistant` class is the unified entry point for all
skforecast-ai functionality. It ties together:
1. **Data profiling** — `inspect()`
2. **Recommendation** — `recommend()`
3. **Code generation** — `generate_code()`
4. **LLM-powered Q&A** — `ask()` (requires LLM)
5. **Plan explanation** — `explain()` (requires LLM)

## Tier System

| Tier | LLM Required | Methods Available |
|------|-------------|-------------------|
| **Tier 0** | No (`llm=None`) | `inspect`, `recommend`, `generate_code` |
| **Tier 1/2** | Yes (`llm="provider:model"`) | All methods including `ask`, `explain` |

## Setup

In [1]:
import skforecast_ai
from skforecast_ai import (
    AskResult,
    ForecastingAssistant,
    GenerateResult,
    LLMRequiredError,
    RecommendResult,
)

print(f"skforecast-ai version: {skforecast_ai.__version__}")

skforecast-ai version: 0.1.0


## 1. Sample Data

We create a simple daily sales dataset with an exogenous variable for
demonstration purposes.

In [2]:
import numpy as np
import pandas as pd

dates = pd.date_range("2023-01-01", periods=365, freq="D")
df = pd.DataFrame(
    {
        "date": dates,
        "sales": np.cumsum(np.random.default_rng(42).normal(1, 5, 365)),
        "temperature": 15 + 10 * np.sin(np.linspace(0, 2 * np.pi, 365)),
        "promo": np.random.default_rng(42).choice([0, 1], size=365, p=[0.8, 0.2]),
    }
)
print(f"Shape: {df.shape}")
df.head()

Shape: (365, 4)


,date,sales,temperature,promo
0,2023-01-01,2.523585,15.000000,0
1,2023-01-02,-1.676335,15.172606,0
2,2023-01-03,3.075921,15.345161,1
3,2023-01-04,8.778744,15.517614,0
4,2023-01-05,0.023568,15.689911,0


## 2. Tier 0 — Deterministic Mode (No LLM)

Create an assistant without an LLM. All deterministic methods work out of
the box with zero external dependencies.

In [3]:
assistant = ForecastingAssistant()

print(f"LLM: {assistant.llm}")
print(f"send_data_to_llm: {assistant.send_data_to_llm}")

LLM: None
send_data_to_llm: False


### 2.1 `inspect()` — Profile the dataset

In [4]:
profile = assistant.inspect(data=df, target="sales", date_column="date")

print(f"Type: {type(profile).__name__}")
print(f"Observations: {profile.n_observations}")
print(f"Series: {profile.n_series}")
print(f"Frequency: {profile.frequency}")
print(f"Index type: {profile.index_type}")
print(f"Exog columns: {profile.exog_columns}")
print(f"Categorical exog: {profile.categorical_exog}")
print(f"Seasonalities: {profile.inferred_seasonalities}")
print(f"Warnings: {profile.warnings}")

Type: DataProfile
Observations: 365
Series: 1
Frequency: D
Index type: datetime
Exog columns: ['temperature', 'promo']
Categorical exog: []
Seasonalities: [7, 365]
Warnings: []


### 2.2 `recommend()` — Generate a forecasting plan

In [5]:
result = assistant.recommend(data=df, target="sales", date_column="date", horizon=30)

print(f"Type: {type(result).__name__}")
assert isinstance(result, RecommendResult)

plan = result.plan
print(f"\n--- Forecast Plan ---")
print(f"Task type: {plan.task_type}")
print(f"Forecaster: {plan.forecaster}")
print(f"Estimator: {plan.estimator}")
print(f"Horizon: {plan.horizon}")
print(f"Lags: {plan.lags}")
print(f"Metric: {plan.metric}")
print(f"Backtesting: {plan.backtesting_strategy}")
print(f"Interval method: {plan.interval_method}")
print(f"Use exog: {plan.use_exog}")
print(f"Rationale: {plan.rationale}")

Type: RecommendResult

--- Forecast Plan ---
Task type: single_series
Forecaster: ForecasterRecursive
Estimator: Ridge
Horizon: 30
Lags: [1, 2, 3, 4, 5, 6, 7]
Metric: mean_absolute_error
Backtesting: TimeSeriesFold
Interval method: bootstrapping
Use exog: True
Rationale: A single-series ML forecaster (ForecasterRecursive) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via bootstrapping. Exogenous variables detected: ['temperature', 'promo'].


### 2.3 `generate_code()` — Produce a complete forecasting script

In [6]:
result = assistant.generate_code(
    data=df, target="sales", date_column="date", horizon=30
)

print(f"Type: {type(result).__name__}")
assert isinstance(result, GenerateResult)

print(f"Profile target: {result.profile.target}")
print(f"Plan forecaster: {result.plan.forecaster}")
print(f"Code length: {len(result.code)} characters")
print(f"\n{'=' * 60}")
print(result.code)

Type: GenerateResult
Profile target: sales
Plan forecaster: ForecasterRecursive
Code length: 1510 characters

import pandas as pd
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from sklearn.linear_model import Ridge

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Train/test split (time-based)
n_train = int(len(data) * 0.8)
y_train = data.iloc[:n_train]['sales']
y_test = data.iloc[n_train:]['sales']
exog_train = data.iloc[:n_train][['temperature', 'promo']]
exog_test = data.iloc[n_train:][['temperature', 'promo']]

# Create forecaster
forecaster = ForecasterRecursive(
    estimator = Ridge(random_state=123),
    lags      = [1, 2, 3, 4, 5, 6, 7],
)

# Fit
forecaster.fit(y=y_train, exog=exog_train)

# Predict
predictions = forecaster.predict(steps=30, exog=exog_test.iloc[:30])
print(predictions)

# Backtesting
cv = TimeSeriesFold(
    steps            

### 2.4 Accepting CSV paths

The assistant also accepts file paths (str or Path) instead of DataFrames.

In [7]:
import tempfile
from pathlib import Path

with tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w") as f:
    df.to_csv(f, index=False)
    csv_path = f.name

profile_from_csv = assistant.inspect(data=csv_path, target="sales", date_column="date")
print(f"From CSV — Observations: {profile_from_csv.n_observations}")
print(f"From CSV — Frequency: {profile_from_csv.frequency}")
print(f"From CSV — Exog: {profile_from_csv.exog_columns}")

Path(csv_path).unlink()

From CSV — Observations: 365
From CSV — Frequency: None
From CSV — Exog: ['temperature', 'promo']


### 2.5 Passing kwargs to the recommendation engine

The `recommend()` and `generate_code()` methods forward extra keyword
arguments to `recommend_plan()`. For example, you can request a
statistical or foundation model preference.

In [8]:
# Prefer statistical models
result_stat = assistant.recommend(
    data=df, target="sales", date_column="date", horizon=30,
    prefer_statistical=True,
)
print(f"Statistical preference:")
print(f"  Forecaster: {result_stat.plan.forecaster}")
print(f"  Task type: {result_stat.plan.task_type}")

# Prefer foundation models
result_found = assistant.recommend(
    data=df, target="sales", date_column="date", horizon=30,
    prefer_foundation=True,
)
print(f"\nFoundation preference:")
print(f"  Forecaster: {result_found.plan.forecaster}")
print(f"  Task type: {result_found.plan.task_type}")

Statistical preference:
  Forecaster: ForecasterStats
  Task type: statistical

Foundation preference:
  Forecaster: ForecasterFoundation
  Task type: foundation


## 3. LLMRequiredError — Guard for LLM-only methods

When `llm=None` (Tier 0), calling `ask()` or `explain()` raises a
clear `LLMRequiredError` instead of failing silently.

In [9]:
try:
    assistant.ask("Forecast my sales 30 days ahead")
except LLMRequiredError as e:
    print(f"LLMRequiredError: {e}")

LLMRequiredError: `ask()` requires an LLM. Pass `llm=...` when creating ForecastingAssistant.


In [10]:
try:
    assistant.explain(plan=result.plan)
except LLMRequiredError as e:
    print(f"LLMRequiredError: {e}")

LLMRequiredError: `explain()` requires an LLM. Pass `llm=...` when creating ForecastingAssistant.


## 4. Tier 1/2 — With LLM (using TestModel mock)

For demonstration without requiring API keys, we use Pydantic AI's `TestModel`
which returns deterministic mock responses.

In [11]:
from pydantic_ai.models.test import TestModel

# Create an assistant with an LLM configured
assistant_llm = ForecastingAssistant(
    llm="openai:gpt-4o-mini",
    send_data_to_llm=False,
)

# Inject TestModel to avoid real API calls
assistant_llm._model = TestModel()

print(f"LLM: {assistant_llm.llm}")
print(f"send_data_to_llm: {assistant_llm.send_data_to_llm}")

LLM: openai:gpt-4o-mini
send_data_to_llm: False


### 4.1 `recommend()` works identically (deterministic)

Even with an LLM configured, `recommend()` uses the deterministic rule
engine — the LLM does not influence recommendations.

In [12]:
result_llm = assistant_llm.recommend(
    data=df, target="sales", date_column="date", horizon=30
)

# Results are identical to Tier 0
print(f"Tier 0 plan == Tier 1 plan: {result.plan == result_llm.plan}")
print(f"Forecaster: {result_llm.plan.forecaster}")
print(f"Horizon: {result_llm.plan.horizon}")

Tier 0 plan == Tier 1 plan: True
Forecaster: ForecasterRecursive
Horizon: 30


### 4.2 `explain()` — LLM explains the plan

With a real LLM, this would return a human-readable explanation. With
`TestModel`, it returns a mock string response.

In [13]:
explanation = assistant_llm.explain(plan=result_llm.plan, profile=result_llm.profile)

print(f"Type: {type(explanation).__name__}")
print(f"Explanation: {explanation}")

RuntimeError: This event loop is already running

## 5. Result Schema Serialization

All result objects are Pydantic models and can be serialized to JSON for
storage, logging, or API responses.

In [14]:
import json

# RecommendResult to JSON
recommend_json = result.model_dump_json(indent=2)
print(f"RecommendResult JSON (first 500 chars):\n{recommend_json[:500]}...")

RecommendResult JSON (first 500 chars):
{
  "profile": {
    "n_observations": 365,
    "n_series": 1,
    "index_type": "datetime",
    "frequency": "D",
    "target": "sales",
    "date_column": "date",
    "series_id_column": null,
    "exog_columns": [
      "temperature",
      "promo"
    ],
    "categorical_exog": [],
    "missing_values": {},
    "inferred_seasonalities": [
      7,
      365
    ],
    "warnings": []
  },
  "plan": {
    "task_type": "single_series",
    "forecaster": "ForecasterRecursive",
    "estimator": "...


In [15]:
# GenerateResult to dict
gen_result = assistant.generate_code(
    data=df, target="sales", date_column="date", horizon=30
)
gen_dict = gen_result.model_dump()

print(f"Keys: {list(gen_dict.keys())}")
print(f"Code starts with: {gen_dict['code'][:80]}...")

Keys: ['profile', 'plan', 'code']
Code starts with: import pandas as pd
from skforecast.recursive import ForecasterRecursive
from sk...


## 6. Multi-Series Example

The assistant handles multi-series datasets just as easily.

In [16]:
# Create a multi-series dataset
multi_dates = pd.date_range("2023-01-01", periods=200, freq="D")
df_multi = pd.DataFrame(
    {
        "date": np.tile(multi_dates, 3),
        "series_id": np.repeat(["store_A", "store_B", "store_C"], 200),
        "revenue": np.random.default_rng(123).normal(100, 20, 600).cumsum(),
        "ads_spend": np.random.default_rng(456).uniform(0, 50, 600),
    }
)
print(f"Multi-series shape: {df_multi.shape}")
df_multi.head()

Multi-series shape: (600, 4)


,date,series_id,revenue,ads_spend
0,2023-01-01,store_A,80.217573,23.483662
1,2023-01-02,store_A,172.861840,22.382765
2,2023-01-03,store_A,298.620345,6.202840
3,2023-01-04,store_A,402.499834,37.742499
4,2023-01-05,store_A,520.904452,48.649423


In [17]:
result_multi = assistant.recommend(
    data=df_multi,
    target="revenue",
    date_column="date",
    series_id_column="series_id",
    horizon=14,
)

print(f"Task type: {result_multi.plan.task_type}")
print(f"Forecaster: {result_multi.plan.forecaster}")
print(f"Series detected: {result_multi.profile.n_series}")
print(f"Exog: {result_multi.profile.exog_columns}")
print(f"Rationale: {result_multi.plan.rationale}")

Task type: multi_series
Forecaster: ForecasterRecursiveMultiSeries
Series detected: 3
Exog: ['ads_spend']
Rationale: The dataset contains 3 series, so a multi-series forecaster (ForecasterRecursiveMultiSeries) is recommended. The estimator is LGBMRegressor. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via conformal. Exogenous variables detected: ['ads_spend'].


In [18]:
# Generate multi-series code
result_multi_code = assistant.generate_code(
    data=df_multi,
    target="revenue",
    date_column="date",
    series_id_column="series_id",
    horizon=14,
)
print(result_multi_code.code)

import pandas as pd
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.model_selection import backtesting_forecaster_multiseries, TimeSeriesFold
from lightgbm import LGBMRegressor

# Load data (long format)
data = pd.read_csv('data.csv', parse_dates=['date'])

# Pivot to wide format (columns = series)
series = data.pivot_table(
    index='date', columns='series_id', values='revenue'
)
series.index = pd.DatetimeIndex(series.index)
series.index.name = None

# Train/test split
n_train = int(len(series) * 0.8)

# Create forecaster
forecaster = ForecasterRecursiveMultiSeries(
    estimator = LGBMRegressor(random_state=123),
    lags      = [1, 2, 3, 4, 5, 6, 7],
    encoding  = 'ordinal',
)

# Fit
forecaster.fit(series=series)

# Predict
predictions = forecaster.predict(steps=14)
print(predictions)

# Backtesting
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = n_train,
    refit              = False,
)

metric, predictions_bt = backtes

## Summary

| Method | Tier 0 (no LLM) | Tier 1/2 (with LLM) |
|--------|-----------------|---------------------|
| `inspect()` | Deterministic profiling | Same |
| `recommend()` | Deterministic rules | Same (LLM not used) |
| `generate_code()` | Deterministic templates | Same |
| `ask()` | Raises `LLMRequiredError` | LLM parses intent → tools |
| `explain()` | Raises `LLMRequiredError` | LLM explains plan |

The `ForecastingAssistant` provides a clean, unified API where:
- **Privacy is preserved** by default (`send_data_to_llm=False`)
- **Deterministic methods always work** without external dependencies
- **LLM methods fail explicitly** with actionable error messages
- **Results are structured** Pydantic models, easily serializable